# Análisis de NASDAQ-100

## 1. Investigación y Planteamiento del Problema

### Contexto
El **NASDAQ-100** es uno de los índices bursátiles más relevantes a nivel global, compuesto por las 100 empresas no financieras más grandes que cotizan en el mercado Nasdaq, con un fuerte predominio del sector tecnológico y de innovación (como Apple, Microsoft, NVIDIA, Amazon y Alphabet).

En el análisis financiero, comprender la **estacionalidad** (patrones recurrentes en los rendimientos de un activo según la época del año) permite evaluar si existen ventanas históricas de oportunidad o de mayor riesgo. Dentro de la literatura financiera destacan fenómenos conocidos como:
* **Efecto Diciembre / *Santa Claus Rally*:** Tendencia histórica observada en los mercados accionarios donde los precios suelen mostrar un repunte durante las últimas semanas de diciembre y principios de enero, impulsado por ajustes de portafolios institucionales, optimismo de fin de año y menor volumen de ventas fiscales (*tax-loss harvesting*).
* **Estacionalidad mensual (*Sell in May and go away* / caídas en septiembre):** Anomalías de calendario que sugieren que ciertos meses presentan rendimientos promedio consistentemente negativos o mayor volatilidad.

### Objetivo de la Investigación
Reducir el universo de inversión al conjunto de empresas que integran el **NASDAQ-100** para analizar su comportamiento histórico diario y determinar:
1. ¿Existe un mes promedio en el que las acciones del NASDAQ-100 presenten caídas o rendimientos negativos de forma consistente?
2. ¿Se cumple el "efecto diciembre" (incremento generalizado en los precios/rendimientos) dentro de las acciones que componen este índice?


## 2. Exploración de los Datos

Para comenzar con la exploración de los datos en el dataset me voy a ayudar de la sugerencia de código propia de kaggle para la descarga de los datos más recientes.

El dataset utilizado para este análisis se encuentra en: https://www.kaggle.com/datasets/svaningelgem/nasdaq-100-daily-stock-prices?select=AMD.csv

In [ ]:
import kagglehub

ruta_dataset = kagglehub.dataset_download("svaningelgem/nasdaq-100-daily-stock-prices")
print(ruta_dataset)

C:\Users\Diana\.cache\kagglehub\datasets\svaningelgem\nasdaq-100-daily-stock-prices\versions\839


### Conteo de archivos descargados

Antes de revisar la estructura de los datos, se verifica cuántos archivos CSV quedaron disponibles en la carpeta descargada desde Kaggle para confirmar que el conjunto esté completo.

In [9]:
import glob
import os

from django.core import files
ruta_archivos = os.path.join(ruta_dataset, "*.csv")
archivos = glob.glob(ruta_archivos)
print(f"Se encontraron {len(archivos)} archivos CSV.")

Se encontraron 100 archivos CSV.


### Unión y Análisis de los Datos

Ya que se verificó la cantidad de archivos descargados, en esta sección se importan los 100 archivos CSV individuales del dataset para juntarlos en un único dataframe que se estará ocupando para el análisis:

In [10]:
import glob
import os
import pandas as pd

lista_df = []
for archivo in archivos:
    df_archivo = pd.read_csv(archivo)
    lista_df.append(df_archivo)

df_nasdaq = pd.concat(lista_df, ignore_index=True)

display(df_nasdaq.head())
print("\nDimensiones:", df_nasdaq.shape)
print("\nColumnas:\n", df_nasdaq.dtypes)

,ticker,date,open,high,low,close
0,AAPL,1980-12-12,0.13,0.13,0.13,0.13
1,AAPL,1980-12-15,0.12,0.12,0.12,0.12
2,AAPL,1980-12-16,0.11,0.11,0.11,0.11
3,AAPL,1980-12-17,0.12,0.12,0.12,0.12
4,AAPL,1980-12-18,0.12,0.12,0.12,0.12



Dimensiones: (714762, 6)

Columnas:
 ticker        str
date          str
open      float64
high      float64
low       float64
close     float64
dtype: object



Al inspeccionar el dataset, de primera instancia se observó que está organizado en múltiples archivos CSV (uno por acción) con registros históricos diarios de cotización, cada archivo manejan las mismas columas que son las siguientes:

* **`ticker`:** Las siglas o clave que identifican de forma única a cada empresa o acción dentro del mercado (por ejemplo, `FANG`, `AAPL`, `MSFT`).
* **`date`:** La fecha correspondiente al día de cotización bursátil del registro.
* **`open`:** El precio de apertura; el valor al que cotizó la acción al inicio de la jornada en el mercado.
* **`high`:** El precio máximo alcanzado por la acción durante el transcurso de esa sesión o día.
* **`low`:** El precio mínimo al que descendió la acción durante ese mismo día.
* **`close`:** El precio de cierre; el último valor registrado al finalizar la jornada bursátil (base habitual para calcular rendimientos).

Para terminar la exploración inicial se revisará qué rango de fechas cubre cada acción y cuántos registros tiene disponible, ya que no todas las empresas se unieron a NASDAQ en el mismo momento.

In [11]:
rango_fechas = (
    df_nasdaq.groupby("ticker")["date"]
    .agg(fecha_inicio="min", fecha_fin="max", total_registros="count")
    .reset_index()
)

print("Primeras acciones según fecha de inicio:")
display(rango_fechas.sort_values("fecha_inicio").head(10))

print("\nÚltimas acciones según fecha de inicio:")
display(rango_fechas.sort_values("fecha_inicio", ascending=False).head(10))

Primeras acciones según fecha de inicio:


,ticker,fecha_inicio,fecha_fin,total_registros
6,AEP,1962-01-02,2026-11-18,16330
40,HON,1962-01-02,2026-11-18,16330
44,INTC,1972-01-07,2026-11-18,13824
98,WMT,1972-03-20,2026-11-18,13764
94,TXN,1972-06-01,2026-11-18,13731
76,PEP,1972-06-01,2026-11-18,13731
99,XEL,1973-02-21,2026-11-18,13551
32,EXC,1973-05-02,2026-11-18,13502
97,WDC,1978-10-31,2026-11-18,12112
74,PCAR,1980-03-17,2026-11-18,11765



Últimas acciones según fecha de inicio:


,ticker,fecha_inicio,fecha_fin,total_registros
85,SKHY,2026-07-10,2026-11-18,93
41,HONA,2026-06-15,2026-11-18,110
54,MDLN,2025-12-17,2026-11-18,232
86,SNDK,2025-02-13,2026-11-18,444
13,ARM,2023-09-14,2026-11-18,799
20,CEG,2022-01-19,2026-11-18,1214
42,HOOD,2021-07-29,2026-11-18,1334
11,APP,2021-04-15,2026-11-18,1407
89,SPCX,2020-12-17,2026-11-18,1328
1,ABNB,2020-12-10,2026-11-18,1492


### Interpretación de las fechas de inicio

Al revisar las fechas de inicio por acción, se observa que no todas las empresas tienen registros desde décadas atrás; algunas cuentan con historial desde los años 60 o 70, mientras que otras inician en años recientes como 2020, 2023 o incluso 2026. Esto no significa necesariamente que el archivo esté corrupto, sino que la empresa cotiza o está registrada desde una fecha posterior como ya se suponía.
Para evitar sesgos al evaluar qué mes tiende a bajar y cómo se comporta en específico el mes de diciembre, es importante revisar qué acciones tienen suficiente historial con el que se pueda trabajar, ya que por ejemplo, empresas como SKHY que apenas se integraron en 2026, no tienen registros para el mes de Diciembre o hay empresas que apenas tienen un par de años de registros.

### Conteo de registros por acción y cálculo de cuartiles

Para entender cómo se distribuye la cantidad de datos históricos entre las acciones del NASDAQ-100, primero contamos cuántos registros tiene cada `ticker` y luego calculamos sus cuartiles. Esto ayuda a identificar qué acciones tienen poca historia reciente frente a las que tienen series más largas antes de tomar decisiones de filtrado o análisis.

In [13]:
conteo_por_ticker = (
    df_nasdaq.groupby("ticker")
    .agg(total_registros=("date", "count"))
    .reset_index()
)

cuartil_uno = conteo_por_ticker["total_registros"].quantile(0.25)
mediana_registros = conteo_por_ticker["total_registros"].quantile(0.50)
cuartil_tres = conteo_por_ticker["total_registros"].quantile(0.75)
rango_intercuartilico = cuartil_tres - cuartil_uno

limite_inferior_registros = cuartil_uno - 1.5 * rango_intercuartilico
limite_superior_registros = cuartil_tres + 1.5 * rango_intercuartilico

display(conteo_por_ticker.head())
print(f"Q1: {cuartil_uno:,.0f} registros")
print(f"Q2 (mediana): {mediana_registros:,.0f} registros")
print(f"Q3: {cuartil_tres:,.0f} registros")
print(f"IQR: {rango_intercuartilico:,.0f} registros")
print(f"Límite inferior: {limite_inferior_registros:,.0f} registros")
print(f"Límite superior: {limite_superior_registros:,.0f} registros")

,ticker,total_registros
0,AAPL,11578
1,ABNB,1492
2,ADBE,10145
3,ADI,11765
4,ADP,11765


Q1: 3,545 registros
Q2 (mediana): 7,041 registros
Q3: 10,337 registros
IQR: 6,792 registros
Límite inferior: -6,643 registros
Límite superior: 20,525 registros
